# 🧠 Notebook 1 — Download & Pré-processamento

Parte 1 de 4 do pipeline de predição de crises.

**Datasets:** CHB-MIT, Siena, **Mendeley (AUB)** e SeizeIT2.

Este notebook: baixa só os arquivos com crise, lê as anotações, **reamostra tudo para um fs comum**, filtra (notch + passa-banda) e salva o **Nível 1 (L1)** — sinal filtrado contínuo + labels amostra-a-amostra + nomes dos canais. Os notebooks seguintes leem o L1 do disco.

## 1.1 Imports

In [1]:
%pip install numpy pandas matplotlib seaborn mne PyWavelets boto3 scipy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, json, io, warnings, gc, html
import html.parser
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import pywt
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, welch, resample_poly
from scipy.stats  import kurtosis as sp_kurtosis, skew as sp_skew
from tqdm.auto    import tqdm

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
print("✅ Imports OK")

✅ Imports OK


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2 Configuração global

Usada por todos os 4 notebooks (cada um é independente e redefine a config no topo).

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Diretórios ───────────────────────────────────
ROOT_DIR    = 'data'
CHBMIT_DIR  = os.path.join(ROOT_DIR, 'chb-mit')
SIENA_DIR   = os.path.join(ROOT_DIR, 'siena')
SEIZEIT_DIR = os.path.join(ROOT_DIR, 'seizeit2')
MENDELEY_DIR= os.path.join(ROOT_DIR, 'mendeley')
L1_DIR      = os.path.join(ROOT_DIR, 'level1_signals')
L2_DIR      = os.path.join(ROOT_DIR, 'level2_windows')
FEAT_DIR    = os.path.join(ROOT_DIR, 'features')   # subpastas por nível: features/<level>/
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')
for d in [CHBMIT_DIR, SIENA_DIR, SEIZEIT_DIR, MENDELEY_DIR,
          L1_DIR, L2_DIR, FEAT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Filtragem ──────────────────────────────────
F_HP, F_LP, F_ORDER = 0.5, 40.0, 4
NOTCH = {'CHBMIT': 60.0, 'Siena': 50.0, 'SeizeIT2': 50.0, 'Mendeley': 50.0}

# Reamostragem comum (Siena=512, Mendeley=500, demais=256) → fs único para
# tornar features espectrais comparáveis e permitir juntar datasets.
TARGET_FS = 256

# ── Janelamento ────────────────────────────────
WIN_SEC, OVERLAP = 4, 0.50

# ── Rotulagem ──────────────────────────────────
# PRE_SEC vem do NB2.5 (estimado empiricamente via PELT). Fallback = 10 min
# se o estimador ainda não rodou. POST/IGAP seguem fixos (não são alvo de predição).
_PREICTAL_JSON = os.path.join(ROOT_DIR, 'preictal_estimate.json')
if os.path.exists(_PREICTAL_JSON):
    PRE_SEC = int(json.load(open(_PREICTAL_JSON))['PRE_SEC_ESTIMATED'])
    print(f"ℹ️  PRE_SEC carregado do NB2.5: {PRE_SEC}s ({PRE_SEC/60:.1f} min)")
else:
    PRE_SEC = 10*60
    print(f"⚠️  preictal_estimate.json não encontrado — usando PRE_SEC fallback = 10 min."
          f" Rode o NB2.5 e re-execute para usar a janela estimada.")
POST_SEC, IGAP_SEC = 10*60, 10*60
# Pré-ictal mínimo: se a janela disponível antes do onset for menor que isso,
# a crise NÃO recebe rótulo pré-ictal (evita pré curto demais que confunde o modelo)
MIN_PRE_SEC = 3*60
LBL = dict(interictal=0, preictal=1, ictal=2, postictal=3, unknown=-1)
LBL_NAMES = {v: k for k, v in LBL.items()}

# ── Cap de interictal na extração (controla custo do SeizeIT2) ──────────
MAX_INTER_RATIO = 10   # máx. interictal:eventos por paciente (None desliga)

# ── PACIENTES POR DATASET ──────────────────────────────
# Mendeley: todos os 6 pacientes (todas as 20 gravações têm crise)
# CHB-MIT, Siena, SeizeIT2: 6 pacientes cada (mesmo número do Mendeley)
PATIENTS = {
    'CHBMIT'  : ['chb01', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07'],
    'Siena'   : ['PN00', 'PN01', 'PN03', 'PN05', 'PN06', 'PN07'],
    'Mendeley': ['p10', 'p11', 'p12', 'p13', 'p14', 'p15'],  # todos os 6
    'SeizeIT2': ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006'],
}
PILOT = {k: v[0] for k, v in PATIENTS.items()}
SEIZEIT_SESSION = 'ses-01'


# ── NÍVEIS DE REDUÇÃO DE CANAIS = subconjuntos de REGIÕES cerebrais ────────
# Por que regiões e não eletrodos fixos? Os datasets usam montagens diferentes
# (CHB-MIT é bipolar, Siena/Mendeley referencial, SeizeIT2 behind-the-ear), então
# "o mesmo eletrodo" não existe entre todos. Mapear cada canal → região e agregar
# por região (1) funciona em qualquer montagem, (2) dá vetor de tamanho fixo para
# poder juntar datasets, e (3) mantém a interpretabilidade (SHAP por região).
REGIONS = ['frontal', 'temporal', 'central', 'parietal', 'occipital']

LEVELS = {
    'R5': ['frontal', 'temporal', 'central', 'parietal', 'occipital'],  # completo (~19 ch)
    'R3': ['frontal', 'temporal', 'central'],                           # reduzido (~8 ch)
    'R2': ['frontal', 'temporal'],                                      # vestível (~4 ch)
    'R1': ['temporal'],                                                 # ultra-compacto / SeizeIT2
}
# Quais datasets participam de cada nível. SeizeIT2 (behind-the-ear, ~região
# temporal) só entra no nível mínimo R1, que é onde ele faz sentido físico.
LEVEL_DATASETS = {
    'R5': ['CHBMIT', 'Siena', 'Mendeley'],
    'R3': ['CHBMIT', 'Siena', 'Mendeley'],
    'R2': ['CHBMIT', 'Siena', 'Mendeley'],
    'R1': ['CHBMIT', 'Siena', 'Mendeley', 'SeizeIT2'],
}

# ── Mapa eletrodo → região (sistema 10-20; aceita variantes T7/T3 etc.) ────
ELECTRODE_REGION = {}
for _e in ['fp1','fp2','f7','f3','fz','f4','f8','af3','af4']:           ELECTRODE_REGION[_e]='frontal'
for _e in ['t3','t4','t5','t6','t7','t8','p7','p8','ft9','ft10','tp7','tp8']: ELECTRODE_REGION[_e]='temporal'
for _e in ['c3','cz','c4','fc1','fc2','cp1','cp2']:                     ELECTRODE_REGION[_e]='central'
for _e in ['p3','pz','p4','po3','po4']:                                 ELECTRODE_REGION[_e]='parietal'
for _e in ['o1','o2','oz']:                                             ELECTRODE_REGION[_e]='occipital'

print("✅ Configurações OK")
print(f"   fs alvo     : {TARGET_FS} Hz | Janela {WIN_SEC}s overlap {int(OVERLAP*100)}%")
print(f"   Pré/Pós     : {PRE_SEC//60}/{POST_SEC//60} min | cap interictal {MAX_INTER_RATIO}:1")
print(f"   Níveis      : " + " | ".join(f"{k}({len(v)} reg)" for k,v in LEVELS.items()))
for k, v in PATIENTS.items():
    print(f"   {k:9s}: {v}")

⚠️  preictal_estimate.json não encontrado — usando PRE_SEC fallback = 10 min. Rode o NB2.5 e re-execute para usar a janela estimada.
✅ Configurações OK
   fs alvo     : 256 Hz | Janela 4s overlap 50%
   Pré/Pós     : 10/10 min | cap interictal 10:1
   Níveis      : R5(5 reg) | R3(3 reg) | R2(2 reg) | R1(1 reg)
   CHBMIT   : ['chb01', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07']
   Siena    : ['PN00', 'PN01', 'PN03', 'PN05', 'PN06', 'PN07']
   Mendeley : ['p10', 'p11', 'p12', 'p13', 'p14', 'p15']
   SeizeIT2 : ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006']


In [4]:
def normalize_electrode(name):
    """Limpa um nome de canal: 'EEG Fp1-REF' -> 'fp1'. Para bipolar 'F7-T3'
    devolve o PRIMEIRO nó ('f7'), que define a região do canal."""
    s = str(name).lower()
    s = s.replace('eeg', ' ').replace('-ref', ' ').replace('-le', ' ').replace('ref', ' ')
    s = s.strip()
    # bipolar 'a-b' -> primeiro nó
    if '-' in s:
        s = s.split('-')[0]
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

def channel_region(name, dataset=None):
    """Região cerebral de um canal. SeizeIT2 (behind-the-ear) -> temporal."""
    if dataset == 'SeizeIT2':
        return 'temporal'
    return ELECTRODE_REGION.get(normalize_electrode(name), None)

def regions_present(ch_names, dataset=None):
    """Dict região -> lista de índices de canais naquela região."""
    out = {r: [] for r in REGIONS}
    for i, nm in enumerate(ch_names):
        r = channel_region(nm, dataset)
        if r in out:
            out[r].append(i)
    return out

## 2 — Download dos datasets (só arquivos com crise)

### 2.1 CHB-MIT (S3 PhysioNet)

In [5]:
def s3_client():
    return boto3.client('s3', region_name='us-east-1',
                        config=Config(signature_version=UNSIGNED))

def download_chbmit_patient(patient, local_root=CHBMIT_DIR):
    """
    Ignora o 'RECORDS-WITH-SEIZURES' falho. Lê direto do summary.txt 
    do paciente para garantir que NENHUM arquivo com crise fique de fora.
    """
    s3      = s3_client()
    bucket  = 'physionet-open'
    prefix  = 'chbmit/1.0.0/'
    pat_dir = os.path.join(local_root, patient)
    os.makedirs(pat_dir, exist_ok=True)

    # 1. Baixa o summary.txt primeiro
    sum_key   = f'{prefix}{patient}/{patient}-summary.txt'
    sum_local = os.path.join(pat_dir, f'{patient}-summary.txt')
    if not os.path.exists(sum_local):
        s3.download_file(bucket, sum_key, sum_local)

    # 2. Descobre os EDFs com crise lendo o arquivo diretamente
    seizure_edfs = set()
    with open(sum_local, 'r', encoding='utf-8', errors='ignore') as f:
        blocks = f.read().split('File Name:')
        for block in blocks[1:]:
            lines = block.strip().split('\n')
            fname = lines[0].strip()
            for line in lines:
                if line.startswith('Number of Seizures in File:'):
                    try:
                        n = int(line.split(':')[1].strip())
                        if n > 0:
                            seizure_edfs.add(fname)
                    except: pass

    # 3. Baixa SOMENTE os EDFs com crise identificados
    downloaded = 0
    for fname in tqdm(list(seizure_edfs), desc=f'CHB-MIT {patient}', leave=False):
        local = os.path.join(pat_dir, fname)
        if not os.path.exists(local):
            try:
                s3.download_file(bucket, prefix + patient + '/' + fname, local)
                downloaded += 1
            except Exception as e:
                print(f"  ⚠️  {fname}: {e}")
                
    print(f"✅ CHB-MIT {patient} — {len(seizure_edfs)} EDFs com crise | {downloaded} novos")
    return pat_dir

# ── Baixa todos os pacientes CHB-MIT da lista ──
for _p in PATIENTS['CHBMIT']:
    download_chbmit_patient(_p)

✅ CHB-MIT chb01 — 7 EDFs com crise | 0 novos


✅ CHB-MIT chb03 — 7 EDFs com crise | 0 novos


✅ CHB-MIT chb04 — 3 EDFs com crise | 0 novos


✅ CHB-MIT chb05 — 5 EDFs com crise | 0 novos


✅ CHB-MIT chb06 — 7 EDFs com crise | 0 novos


✅ CHB-MIT chb07 — 3 EDFs com crise | 0 novos


### 2.2 Siena (HTTPS PhysioNet)

In [6]:
import time
SIENA_BASE = 'https://physionet.org/files/siena-scalp-eeg/1.0.0/'

class _LinkParser(html.parser.HTMLParser):
    def __init__(self):
        super().__init__(); self.links = []
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            for attr, val in attrs:
                if attr == 'href' and not val.startswith(('?', '..')):
                    self.links.append(val)

def _siena_seizure_edfs(patient, seizures_list_text):
    """Extrai do Seizures-list-PNxx.txt os nomes de EDF que têm crise (lowercase)."""
    edfs = set()
    for line in seizures_list_text.splitlines():
        m = re.search(r'File name:\s*(\S+\.edf)', line, re.IGNORECASE)
        if m:
            edfs.add(m.group(1).strip().lower())
    return edfs

def _edf_prefix(fname):
    """
    'PN01-2.edf' → 'pn01' | 'PN01.edf' → 'pn01' | 'PN01-1.edf' → 'pn01'
    Também corrige typos comuns no txt do Siena: 'pno6' → 'pn06' (o vs 0).
    """
    prefix = re.sub(r'[-_].*', '', fname.lower().replace('.edf', ''))
    # corrige troca de letra 'o' por dígito '0' e vice-versa nos prefixos
    prefix = re.sub(r'o(\d)', r'0\1', prefix)   # pno6 → pn06
    prefix = re.sub(r'(\D)0', r'\g<1>o', prefix) if not re.search(r'\d', prefix) else prefix
    return prefix

def _download_with_progress(url, local_path, desc=''):
    """Download com barra de progresso em bytes — compatível com Jupyter sem ipywidgets."""
    import sys
    tmp = local_path + '.part'
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            total = int(r.headers.get('Content-Length', 0))
            downloaded = 0
            start = time.time()
            with open(tmp, 'wb') as f:
                while True:
                    chunk = r.read(1024 * 64)   # 64 KB
                    if not chunk:
                        break
                    f.write(chunk)
                    downloaded += len(chunk)
                    elapsed = time.time() - start
                    speed = downloaded / elapsed if elapsed > 0 else 0
                    pct = downloaded / total * 100 if total > 0 else 0
                    done_mb  = downloaded / 1024 / 1024
                    total_mb = total      / 1024 / 1024
                    speed_mb = speed      / 1024 / 1024
                    bar = int(pct / 2)   # 50 chars de largura
                    print(f'\r  {desc}: {pct:5.1f}% |{"█"*bar}{" "*(50-bar)}| '
                          f'{done_mb:.1f}/{total_mb:.1f} MB  {speed_mb:.2f} MB/s',
                          end='', flush=True)
            print()   # quebra linha ao terminar
        os.replace(tmp, local_path)
        return True
    except Exception as e:
        print()   # quebra linha em caso de erro
        if os.path.exists(tmp):
            os.remove(tmp)
        raise e

def download_siena_patient(patient, local_root=SIENA_DIR):
    """
    Baixa, para UM paciente Siena: o Seizures-list (anotações) e APENAS os
    EDFs que aparecem nele (ou seja, com crise). Evita baixar gravações sem crise.
    """
    pat_dir  = os.path.join(local_root, patient)
    os.makedirs(pat_dir, exist_ok=True)
    base_url = SIENA_BASE + patient + '/'

    try:
        with urllib.request.urlopen(base_url, timeout=30) as r:
            content = r.read().decode('utf-8')
    except Exception as e:
        print(f"⚠️  Não foi possível listar {base_url}: {e}")
        return pat_dir

    p = _LinkParser(); p.feed(content)

    # Normaliza links: extrai só o nome do arquivo (trata hrefs absolutos e relativos)
    all_files = []
    for lnk in p.links:
        fname = lnk.rstrip('/').split('/')[-1]
        if fname:
            all_files.append(fname)

    # 1) baixa primeiro os .txt (anotações) para saber quais EDFs têm crise
    txt_files = [l for l in all_files if l.lower().endswith('.txt')]
    for fname in txt_files:
        local = os.path.join(pat_dir, fname)
        if not os.path.exists(local):
            try: urllib.request.urlretrieve(base_url + fname, local)
            except Exception as e: print(f"  ⚠️  {fname}: {e}")

    # 2) descobre EDFs com crise a partir do Seizures-list-PNxx.txt
    seizure_edfs = set()   # lowercase
    sl_path = os.path.join(pat_dir, f'Seizures-list-{patient}.txt')
    if os.path.exists(sl_path):
        with open(sl_path, encoding='utf-8', errors='ignore') as f:
            seizure_edfs = _siena_seizure_edfs(patient, f.read())

    edf_files = [l for l in all_files if l.lower().endswith('.edf')]
    print(f"  ℹ️  {patient}: txt={seizure_edfs} | servidor={edf_files}")

    # O txt pode referenciar 'PN01.edf' enquanto o servidor lista 'PN01-1.edf', 'PN01-2.edf'.
    # Compara só o prefixo do paciente (ex: 'pn01') ignorando o número e o case.
    # Funciona tanto quando o txt tem 'PN01.edf' quanto 'PN01-1.edf'.
    if seizure_edfs:
        sz_prefixes = {_edf_prefix(e) for e in seizure_edfs}
        target_edfs = [e for e in edf_files if _edf_prefix(e) in sz_prefixes]
    else:
        target_edfs = edf_files   # fallback: baixa tudo se txt não foi lido

    if seizure_edfs and not target_edfs:
        print(f"  ⚠️  {patient}: fallback — baixando todos os EDFs do servidor")
        target_edfs = edf_files

    downloaded = 0
    for i, fname in enumerate(target_edfs, 1):
        local = os.path.join(pat_dir, fname)
        if not os.path.exists(local):
            try:
                desc = f'Siena {patient} [{i}/{len(target_edfs)}] {fname}'
                _download_with_progress(base_url + fname, local, desc=desc)
                downloaded += 1
            except Exception as e:
                print(f"  ⚠️  {fname}: {e}")
        else:
            print(f"  ⏭️  Já existe: {fname}")
    print(f"✅ Siena {patient} — {len(target_edfs)} EDFs com crise | {downloaded} novos")
    return pat_dir

for _p in PATIENTS['Siena']:
    download_siena_patient(_p)

  ℹ️  PN00: txt={'pn00-2.edf', 'pn00-5.edf', 'pn00-4.edf', 'pn00-1.edf', 'pn00-3.edf'} | servidor=['PN00-1.edf', 'PN00-2.edf', 'PN00-3.edf', 'PN00-4.edf', 'PN00-5.edf']
  ⏭️  Já existe: PN00-1.edf
  ⏭️  Já existe: PN00-2.edf
  ⏭️  Já existe: PN00-3.edf
  ⏭️  Já existe: PN00-4.edf
  ⏭️  Já existe: PN00-5.edf
✅ Siena PN00 — 5 EDFs com crise | 0 novos
  ℹ️  PN01: txt={'pn01.edf'} | servidor=['PN01-1.edf']
  ⏭️  Já existe: PN01-1.edf
✅ Siena PN01 — 1 EDFs com crise | 0 novos
  ℹ️  PN03: txt={'pn03-2.edf', 'pn03-1.edf'} | servidor=['PN03-1.edf', 'PN03-2.edf']
  ⏭️  Já existe: PN03-1.edf
  ⏭️  Já existe: PN03-2.edf
✅ Siena PN03 — 2 EDFs com crise | 0 novos
  ℹ️  PN05: txt={'pn05-3.edf', 'pn05-4.edf', 'pn05-2.edf'} | servidor=['PN05-2.edf', 'PN05-3.edf', 'PN05-4.edf']
  ⏭️  Já existe: PN05-2.edf
  ⏭️  Já existe: PN05-3.edf
  ⏭️  Já existe: PN05-4.edf
✅ Siena PN05 — 3 EDFs com crise | 0 novos
  ℹ️  PN06: txt={'pno6-1.edf', 'pn06-3.edf', 'pn06-5.edf', 'pno6-2.edf', 'pno6-4.edf'} | servidor=['PN

### 2.3 SeizeIT2 (S3 OpenNeuro)

In [7]:
SEIZEIT_BUCKET  = 'openneuro.org'
SEIZEIT_DATASET = 'ds005873'
SEIZEIT_SESSION = 'ses-01'

# Tipos de evento que SÃO crise no SeizeIT2 (BIDS). Tudo o que não estiver aqui
# (ex.: 'impd' = impedance check, 'bckg' = background) é IGNORADO.
SEIZEIT_SZ_TYPES = {'sz', 'sz_foc_a', 'sz_foc_ia', 'sz_gen', 'szt1', 'szt2'}

def _tsv_has_seizure(tsv_bytes):
    """Lê um _events.tsv (em bytes) e diz se contém ao menos uma crise válida."""
    import io
    try:
        df = pd.read_csv(io.BytesIO(tsv_bytes), sep='\t')
    except Exception:
        return False
    if 'eventType' not in df.columns:
        return False
    et = df['eventType'].astype(str).str.lower().str.strip()
    # crise = começa com 'sz' E não é um tipo claramente não-crise
    is_sz = et.str.startswith('sz')
    return bool(is_sz.any())

def download_seizeit2_subject(subject, local_root=SEIZEIT_DIR):
    """
    Baixa, para UM sujeito SeizeIT2: primeiro os _events.tsv (leves), decide
    quais runs têm crise, e então baixa SOMENTE os _eeg.edf (+ .json) desses runs.
    Evita baixar EDFs grandes de gravações sem crise.
    """
    s3      = s3_client()
    prefix  = f'{SEIZEIT_DATASET}/{subject}/{SEIZEIT_SESSION}/eeg/'
    sub_dir = os.path.join(local_root, subject, SEIZEIT_SESSION, 'eeg')
    os.makedirs(sub_dir, exist_ok=True)

    paginator = s3.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=SEIZEIT_BUCKET, Prefix=prefix):
        keys.extend(o['Key'] for o in page.get('Contents', []))

    # 1) baixa todos os _events.tsv (são pequenos)
    tsv_keys = [k for k in keys if k.endswith('_events.tsv')]
    seizure_runs = set()   # prefixos de run com crise, ex 'sub-001_..._run-03'
    for key in tsv_keys:
        fname = key.split('/')[-1]
        local = os.path.join(sub_dir, fname)
        if not os.path.exists(local):
            try: s3.download_file(SEIZEIT_BUCKET, key, local)
            except Exception as e: print(f"  ⚠️  {fname}: {e}")
        # checa se tem crise
        try:
            with open(local, 'rb') as f:
                if _tsv_has_seizure(f.read()):
                    seizure_runs.add(fname.replace('_events.tsv', ''))
        except Exception:
            pass

    # 2) baixa apenas os _eeg.edf (+ sidecar .json) dos runs com crise
    downloaded = 0
    for key in tqdm(keys, desc=f'SeizeIT2 {subject}', leave=False):
        fname = key.split('/')[-1]
        run_prefix = fname.replace('_eeg.edf','').replace('_eeg.json','')
        is_eeg = fname.endswith('_eeg.edf') or fname.endswith('_eeg.json')
        if is_eeg and run_prefix not in seizure_runs:
            continue   # pula gravações sem crise
        local = os.path.join(sub_dir, fname)
        if not os.path.exists(local):
            try:
                s3.download_file(SEIZEIT_BUCKET, key, local)
                downloaded += 1
            except Exception as e:
                print(f"  ⚠️  {fname}: {e}")
    print(f"✅ SeizeIT2 {subject} — {len(seizure_runs)} runs com crise | {downloaded} novos arquivos")
    return sub_dir

for _p in PATIENTS['SeizeIT2']:
    download_seizeit2_subject(_p)

✅ SeizeIT2 sub-001 — 4 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-002 — 7 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-003 — 1 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-004 — 1 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-005 — 2 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-006 — 1 runs com crise | 0 novos arquivos


### 2.4 Mendeley — Epileptic EEG Dataset (AUB)

Baixado via API pública do Mendeley (cloudscraper contorna o Cloudflare). As anotações vêm numa planilha `Seizures_Information.xlsx` (tempos em fração de dia).

In [8]:
# ── Download Mendeley (Epileptic EEG Dataset, AUB) via API pública ─────────
# Usa cloudscraper para contornar o Cloudflare do data.mendeley.com.
try:
    import cloudscraper
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'cloudscraper', '--quiet'])
    import cloudscraper

MEND_DATASET_ID = '5pc2j46cbc'
MEND_VERSION    = 1
MEND_DOC_FOLDER = '465c0896-7d08-4fbe-8ee3-378beca656d5'
MEND_EDF_FOLDER = '3bc6cc31-0156-490b-8e39-4b740598b289'
_MEND_HEADERS   = {"Accept": "application/vnd.mendeley-public-dataset.1+json"}

def _mendeley_list(folder_id):
    scraper = cloudscraper.create_scraper()
    url = (f"https://data.mendeley.com/public-api/datasets/{MEND_DATASET_ID}"
           f"/files?folder_id={folder_id}&version={MEND_VERSION}")
    r = scraper.get(url, headers=_MEND_HEADERS); r.raise_for_status()
    return r.json()

def download_mendeley(patients=None, local_root=MENDELEY_DIR):
    """Baixa o Seizures_Information.xlsx e os EDFs dos pacientes pedidos.
    patients: lista de prefixos como ['p10','p11']; None = todos os EDFs.
    Cada paciente tem vários *_RecordN.edf; baixamos todos os records do prefixo."""
    scraper = cloudscraper.create_scraper()
    doc_dir = os.path.join(local_root, 'Documentation')
    edf_dir = os.path.join(local_root, 'Raw_EDF_Files')
    os.makedirs(doc_dir, exist_ok=True); os.makedirs(edf_dir, exist_ok=True)

    # 1) planilha de anotações
    for fobj in _mendeley_list(MEND_DOC_FOLDER):
        if fobj['filename'] == 'Seizures_Information.xlsx':
            dst = os.path.join(doc_dir, 'Seizures_Information.xlsx')
            if not os.path.exists(dst):
                print("⬇️  Seizures_Information.xlsx ...")
                with open(dst, 'wb') as f:
                    f.write(scraper.get(fobj['content_details']['download_url']).content)
            else:
                print("  ⏭️  Já existe: Seizures_Information.xlsx")

    # 2) EDFs (filtra pelos prefixos de paciente)
    edf_files = _mendeley_list(MEND_EDF_FOLDER)
    def _keep(fn):
        if patients is None:
            return True
        return any(fn.lower().startswith(p.lower() + '_') or
                   fn.lower().startswith(p.lower() + 'record') for p in patients)
    targets = [f for f in edf_files if f['filename'].lower().endswith('.edf') and _keep(f['filename'])]

    print(f"  ℹ️  Mendeley: {len(targets)} EDFs alvo para {patients}")

    def _download_mendeley_with_progress(url, local_path, desc=''):
        """Download via cloudscraper (contorna Cloudflare) com barra de progresso."""
        tmp = local_path + '.part'
        try:
            r = scraper.get(url, stream=True, timeout=120)
            r.raise_for_status()
            total      = int(r.headers.get('Content-Length', 0))
            downloaded = 0
            start      = time.time()
            with open(tmp, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024 * 64):
                    if not chunk:
                        break
                    f.write(chunk)
                    downloaded += len(chunk)
                    elapsed  = time.time() - start
                    speed    = downloaded / elapsed if elapsed > 0 else 0
                    pct      = downloaded / total * 100 if total > 0 else 0
                    done_mb  = downloaded / 1024 / 1024
                    total_mb = total      / 1024 / 1024
                    speed_mb = speed      / 1024 / 1024
                    bar      = int(pct / 2)
                    print(f'\r  {desc}: {pct:5.1f}% |{"█"*bar}{" "*(50-bar)}| '
                          f'{done_mb:.1f}/{total_mb:.1f} MB  {speed_mb:.2f} MB/s',
                          end='', flush=True)
            print()
            os.replace(tmp, local_path)
            return True
        except Exception as e:
            print()
            if os.path.exists(tmp):
                os.remove(tmp)
            raise e

    dl = 0
    for i, fobj in enumerate(targets, 1):
        dst = os.path.join(edf_dir, fobj['filename'])
        if os.path.exists(dst):
            print(f"  ⏭️  Já existe: {fobj['filename']}")
            continue
        try:
            desc = f"Mendeley [{i}/{len(targets)}] {fobj['filename']}"
            _download_mendeley_with_progress(fobj['content_details']['download_url'], dst, desc=desc)
            dl += 1
        except Exception as e:
            print(f"  ⚠️  {fobj['filename']}: {e}")
    print(f"✅ Mendeley — {len(targets)} EDFs alvo | {dl} novos")

download_mendeley(PATIENTS['Mendeley'])

  ⏭️  Já existe: Seizures_Information.xlsx
  ℹ️  Mendeley: 20 EDFs alvo para ['p10', 'p11', 'p12', 'p13', 'p14', 'p15']
  Mendeley [1/20] p10_Record1.edf: 100.0% |██████████████████████████████████████████████████| 195.8/195.8 MB  4.10 MB/s
  ⏭️  Já existe: p10_Record2.edf
  ⏭️  Já existe: p11_Record1.edf
  ⏭️  Já existe: p11_Record2.edf
  ⏭️  Já existe: p11_Record3.edf
  ⏭️  Já existe: p11_Record4.edf
  ⏭️  Já existe: p12_Record1.edf
  ⏭️  Já existe: p12_Record2.edf
  ⏭️  Já existe: p12_Record3.edf
  ⏭️  Já existe: p13_Record1.edf
  ⏭️  Já existe: p13_Record2.edf
  ⏭️  Já existe: p13_Record3.edf
  ⏭️  Já existe: p13_Record4.edf
  ⏭️  Já existe: p14_Record1.edf
  ⏭️  Já existe: p14_Record2.edf
  ⏭️  Já existe: p14_Record3.edf
  Mendeley [17/20] p15_Record1.edf: 100.0% |██████████████████████████████████████████████████| 232.0/232.0 MB  4.57 MB/s
  ⏭️  Já existe: p15_Record2.edf
  ⏭️  Já existe: p15_Record3.edf
  ⏭️  Já existe: p15_Record4.edf
✅ Mendeley — 20 EDFs alvo | 2 novos


## 3 — Parsers de anotação (intervalos de crise por arquivo)

### 3.1 CHB-MIT

In [9]:
def parse_chbmit_summary(patient_dir, patient):
    """Retorna { 'chb01_03.edf': [(start_s, end_s), ...] } — só arquivos com crise."""
    path = os.path.join(patient_dir, f'{patient}-summary.txt')
    seizures = {}
    cur_file, n_seiz = None, 0
    with open(path, encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if line.startswith('File Name:'):
                cur_file = line.split(':', 1)[1].strip(); n_seiz = 0
            elif line.startswith('Number of Seizures in File:'):
                n_seiz = int(line.split(':', 1)[1].strip())
                if n_seiz > 0: seizures.setdefault(cur_file, [])
            elif cur_file and n_seiz > 0:
                if 'Seizure' in line and 'Start Time' in line:
                    t = int(re.search(r'(\d+)', line.split(':',1)[1]).group(1))
                    seizures[cur_file].append([t, None])
                elif 'Seizure' in line and 'End Time' in line:
                    t = int(re.search(r'(\d+)', line.split(':',1)[1]).group(1))
                    if seizures[cur_file] and seizures[cur_file][-1][1] is None:
                        seizures[cur_file][-1][1] = t
    return {f: [(s,e) for s,e in lst if e is not None]
            for f, lst in seizures.items()
            if any(e is not None for s,e in lst)}

# ── Anotações de TODOS os pacientes CHB-MIT ──────────────────────────────────
# chbmit_ann_all = { 'chb01': { 'chb01_03.edf': [(s,e),...], ... }, ... }
chbmit_ann_all = {}
for _p in PATIENTS['CHBMIT']:
    ann = parse_chbmit_summary(os.path.join(CHBMIT_DIR, _p), _p)
    chbmit_ann_all[_p] = ann
    print(f"✅ CHB-MIT {_p} — {len(ann)} arquivos com crise")
    for fname, seiz in ann.items():
        for s,e in seiz:
            print(f"   {fname}  →  {s}s – {e}s  ({e-s}s)")

✅ CHB-MIT chb01 — 7 arquivos com crise
   chb01_03.edf  →  2996s – 3036s  (40s)
   chb01_04.edf  →  1467s – 1494s  (27s)
   chb01_15.edf  →  1732s – 1772s  (40s)
   chb01_16.edf  →  1015s – 1066s  (51s)
   chb01_18.edf  →  1720s – 1810s  (90s)
   chb01_21.edf  →  327s – 420s  (93s)
   chb01_26.edf  →  1862s – 1963s  (101s)
✅ CHB-MIT chb03 — 7 arquivos com crise
   chb03_01.edf  →  362s – 414s  (52s)
   chb03_02.edf  →  731s – 796s  (65s)
   chb03_03.edf  →  432s – 501s  (69s)
   chb03_04.edf  →  2162s – 2214s  (52s)
   chb03_34.edf  →  1982s – 2029s  (47s)
   chb03_35.edf  →  2592s – 2656s  (64s)
   chb03_36.edf  →  1725s – 1778s  (53s)
✅ CHB-MIT chb04 — 3 arquivos com crise
   chb04_05.edf  →  7804s – 7853s  (49s)
   chb04_08.edf  →  6446s – 6557s  (111s)
   chb04_28.edf  →  1679s – 1781s  (102s)
   chb04_28.edf  →  3782s – 3898s  (116s)
✅ CHB-MIT chb05 — 5 arquivos com crise
   chb05_06.edf  →  417s – 532s  (115s)
   chb05_13.edf  →  1086s – 1196s  (110s)
   chb05_16.edf  →  2317s – 

### 3.2 Siena

In [10]:
def _hms_to_sec(s):
    """Converte 'HH.MM.SS' ou 'HH:MM:SS' para segundos inteiros."""
    s = s.strip().replace(':', '.')
    parts = s.split('.')
    if len(parts) == 3:
        h, m, sec = int(parts[0]), int(parts[1]), int(parts[2])
        return h * 3600 + m * 60 + sec
    return 0

def _clock_offset(reg_start_s, event_s):
    """
    Calcula offset em segundos de event_s em relação a reg_start_s,
    tratando cruzamento de meia-noite (event_s < reg_start_s).
    """
    diff = event_s - reg_start_s
    if diff < 0:          # passou da meia-noite
        diff += 24 * 3600
    return diff

def _normalize_siena_edf_name(fname):
    """
    Normaliza nomes de EDF do Siena que têm typos.
    O txt do Siena às vezes troca o dígito '0' por letra 'O' (ex: 'PNO6-1.edf').
    Esta função converte 'PNO6-1.edf' → 'PN06-1.edf' para bater com o servidor.
    Trata variações: 'pno6', 'PNo6', 'PNO6', 'pN0o6' → todas viram 'PN06'.
    """
    # Caso geral: dentro de um prefixo tipo PN, troca letra 'O'/'o' por '0'
    # quando ela aparece entre 'PN' e um dígito (ou no final do prefixo).
    # Ex: 'PNO6-1.edf' → match 'PNO' + '6' → 'PN0' + '6' = 'PN06'
    def fix(m):
        return m.group(1) + '0' + m.group(2)
    # PN + O/o + dígito
    out = re.sub(r'(PN)[Oo](\d)', fix, fname)
    # Caso o dígito '0' tenha virado letra 'O' em outras posições do prefixo
    # ex: 'PN0o-1.edf' (improvável mas seguro)
    out = re.sub(r'(PN\d)[Oo](\d|[-_.])', lambda m: m.group(1)+'0'+m.group(2), out)
    return out

def _resolve_disk_filename(patient_dir, fname_from_txt):
    """
    Dado o nome do EDF como aparece no txt e o diretório do paciente,
    devolve o nome real que existe no disco.

    Trata 3 casos:
      a) Nome bate exatamente   → retorna o próprio fname
      b) Typo O↔0               → normaliza e tenta de novo
      c) Txt diz 'PN01.edf' mas disco tem 'PN01-1.edf' → procura por prefixo
    """
    if not os.path.isdir(patient_dir):
        return fname_from_txt

    disk_files = [f for f in os.listdir(patient_dir) if f.lower().endswith('.edf')]
    disk_lower = {f.lower(): f for f in disk_files}

    # a) match direto
    if fname_from_txt.lower() in disk_lower:
        return disk_lower[fname_from_txt.lower()]

    # b) normaliza typo O↔0
    norm = _normalize_siena_edf_name(fname_from_txt)
    if norm.lower() in disk_lower:
        return disk_lower[norm.lower()]

    # c) procura por prefixo (txt='PN01.edf' → disco tem 'PN01-1.edf')
    base = os.path.splitext(norm)[0]   # 'PN01'
    candidates = [f for f in disk_files
                  if f.lower().startswith(base.lower() + '-')]
    if candidates:
        return sorted(candidates)[0]   # primeiro EDF começando com 'PN01-'

    # nada bateu — devolve o nome original normalizado para o run_l1 reportar
    return norm

def parse_siena_seizure_list(patient_dir, patient):
    """
    Lê Seizures-list-PNXX.txt e retorna:
        { 'PN00-1.edf': [(start_rel_s, end_rel_s), ...], ... }
    A chave do dict é o NOME REAL DO ARQUIVO NO DISCO (já normalizado
    e mapeado para o que existe na pasta). Tempos são relativos ao início
    da gravação de cada arquivo.
    """
    fname_txt = f'Seizures-list-{patient}.txt'
    path = os.path.join(patient_dir, fname_txt)
    if not os.path.exists(path):
        print(f"⚠️  {fname_txt} não encontrado em {patient_dir}")
        return {}

    seizures   = {}
    cur_file   = None
    reg_start  = None

    with open(path, encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            m = re.match(r'File name:\s*(\S+\.edf)', line, re.IGNORECASE)
            if m:
                # Resolve o nome do arquivo: normaliza typos e ajusta ao que existe no disco
                txt_name  = m.group(1).strip()
                cur_file  = _resolve_disk_filename(patient_dir, txt_name)
                if cur_file.lower() != txt_name.lower():
                    print(f"   ℹ️  {patient}: txt={txt_name!r} → disco={cur_file!r}")
                reg_start = None
                seizures.setdefault(cur_file, [])
                continue

            m = re.match(r'Registration start time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m:
                reg_start = _hms_to_sec(m.group(1))
                continue

            # Aceita "Start time:" e "Seizure start time:" (formato varia entre pacientes)
            m = re.match(r'(?:Seizure\s+)?Start\s+time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m and cur_file is not None and reg_start is not None:
                t = _clock_offset(reg_start, _hms_to_sec(m.group(1)))
                seizures[cur_file].append([t, None])
                continue

            # Aceita "End time:" e "Seizure end time:"
            m = re.match(r'(?:Seizure\s+)?End\s+time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m and cur_file is not None and reg_start is not None:
                t = _clock_offset(reg_start, _hms_to_sec(m.group(1)))
                if seizures[cur_file] and seizures[cur_file][-1][1] is None:
                    seizures[cur_file][-1][1] = t

    return {f: [(s,e) for s,e in lst if e is not None and e > s]
            for f, lst in seizures.items()
            if any(e is not None and e > s for s,e in lst)}

# ── Anotações de TODOS os pacientes Siena ────────────────────────────────────
siena_ann_all = {}
for _p in PATIENTS['Siena']:
    ann = parse_siena_seizure_list(os.path.join(SIENA_DIR, _p), _p)
    siena_ann_all[_p] = ann
    print(f"✅ Siena {_p} — {len(ann)} arquivos com crise")
    for fname, seiz in ann.items():
        for s,e in seiz:
            print(f"   {fname}  →  {s}s – {e}s  ({e-s}s)")


✅ Siena PN00 — 5 arquivos com crise
   PN00-1.edf  →  1143s – 1213s  (70s)
   PN00-2.edf  →  1220s – 1274s  (54s)
   PN00-3.edf  →  765s – 4425s  (3660s)
   PN00-4.edf  →  1006s – 1080s  (74s)
   PN00-5.edf  →  904s – 971s  (67s)
   ℹ️  PN01: txt='PN01.edf' → disco='PN01-1.edf'
✅ Siena PN01 — 1 arquivos com crise
   PN01-1.edf  →  10218s – 10272s  (54s)
   PN01-1.edf  →  46353s – 46427s  (74s)
✅ Siena PN03 — 2 arquivos com crise
   PN03-1.edf  →  38673s – 38784s  (111s)
   PN03-2.edf  →  34921s – 35054s  (133s)
✅ Siena PN05 — 3 arquivos com crise
   PN05-2.edf  →  7163s – 7198s  (35s)
   PN05-3.edf  →  6836s – 6866s  (30s)
   PN05-4.edf  →  3608s – 3647s  (39s)
   ℹ️  PN06: txt='PNO6-1.edf' → disco='PN06-1.edf'
   ℹ️  PN06: txt='PNO6-2.edf' → disco='PN06-2.edf'
   ℹ️  PN06: txt='PNO6-4.edf' → disco='PN06-4.edf'
✅ Siena PN06 — 5 arquivos com crise
   PN06-1.edf  →  5583s – 5647s  (64s)
   PN06-2.edf  →  8860s – 8929s  (69s)
   PN06-3.edf  →  6275s – 6317s  (42s)
   PN06-4.edf  →  5939s 

### 3.3 SeizeIT2

In [11]:
def parse_seizeit2_subject(subject_eeg_dir):
    """
    Varre os *_events.tsv do sujeito e retorna { edf_name: [(onset, onset+dur), ...] }.

    Blindagem contra eventos NÃO-crise: só consideramos linhas cujo eventType
    começa com 'sz' (crise no padrão BIDS do SeizeIT2). Eventos como 'impd'
    (impedance check), 'bckg' (background), movimento etc. são DESCARTADOS,
    pois não começam com 'sz'.
    """
    seizures = {}
    tsv_files = sorted(f for f in os.listdir(subject_eeg_dir)
                       if f.endswith('_events.tsv'))
    for tsv in tsv_files:
        edf_name = tsv.replace('_events.tsv', '_eeg.edf')
        try:
            df = pd.read_csv(os.path.join(subject_eeg_dir, tsv), sep='\t')
        except Exception:
            continue
        if 'eventType' not in df.columns:
            continue
        et = df['eventType'].astype(str).str.lower().str.strip()
        # SÓ crises: eventType começa com 'sz' (descarta impd, bckg, etc.)
        sz = df[et.str.startswith('sz')]
        if sz.empty:
            continue
        pairs = [(float(row['onset']), float(row['onset']) + float(row['duration']))
                 for _, row in sz.iterrows()
                 if float(row.get('duration', 0)) > 0]
        if pairs:
            seizures[edf_name] = pairs
    return seizures

# ── Anotações de TODOS os sujeitos SeizeIT2 ──────────────────────────────────
seizeit_ann_all = {}
for _p in PATIENTS['SeizeIT2']:
    eeg_dir = os.path.join(SEIZEIT_DIR, _p, SEIZEIT_SESSION, 'eeg')
    ann = parse_seizeit2_subject(eeg_dir) if os.path.isdir(eeg_dir) else {}
    seizeit_ann_all[_p] = ann
    print(f"✅ SeizeIT2 {_p} — {len(ann)} arquivos com crise")
    for fname, seiz in ann.items():
        for s,e in seiz:
            print(f"   {os.path.basename(fname)}  →  {s:.1f}s – {e:.1f}s  ({e-s:.1f}s)")

✅ SeizeIT2 sub-001 — 4 arquivos com crise
   sub-001_ses-01_task-szMonitoring_run-03_eeg.edf  →  57975.0s – 58047.0s  (72.0s)
   sub-001_ses-01_task-szMonitoring_run-05_eeg.edf  →  60968.0s – 61049.0s  (81.0s)
   sub-001_ses-01_task-szMonitoring_run-07_eeg.edf  →  16300.0s – 16443.0s  (143.0s)
   sub-001_ses-01_task-szMonitoring_run-08_eeg.edf  →  28285.0s – 28310.0s  (25.0s)
✅ SeizeIT2 sub-002 — 7 arquivos com crise
   sub-002_ses-01_task-szMonitoring_run-01_eeg.edf  →  4046.0s – 4066.0s  (20.0s)
   sub-002_ses-01_task-szMonitoring_run-01_eeg.edf  →  10629.0s – 10810.0s  (181.0s)
   sub-002_ses-01_task-szMonitoring_run-02_eeg.edf  →  5224.0s – 5336.0s  (112.0s)
   sub-002_ses-01_task-szMonitoring_run-02_eeg.edf  →  13745.0s – 13925.0s  (180.0s)
   sub-002_ses-01_task-szMonitoring_run-03_eeg.edf  →  31537.0s – 31589.0s  (52.0s)
   sub-002_ses-01_task-szMonitoring_run-05_eeg.edf  →  69796.0s – 69949.0s  (153.0s)
   sub-002_ses-01_task-szMonitoring_run-05_eeg.edf  →  74529.0s – 74593.0s 

### 3.4 Mendeley (planilha XLSX)

In [12]:
import datetime

def _time_to_sec(x):
    """
    Converte campos de hora da planilha Mendeley para segundos.
    O pandas lê como datetime.time — não como fração de dia.
    """
    if isinstance(x, datetime.time):
        return x.hour * 3600 + x.minute * 60 + x.second
    if isinstance(x, (int, float)):
        return float(x) * 86400   # fração de dia (fallback)
    return None

def parse_mendeley_xlsx(xlsx_path, patients=None):
    """
    Lê Seizures_Information.xlsx e devolve:
        { 'p10': { 'p10_Record1.edf': [(onset_s, offset_s), ...] }, ... }
    Os tempos são lidos como datetime.time pelo pandas e convertidos para
    segundos relativos ao início do arquivo (File onset).
    """
    raw = pd.read_excel(xlsx_path, header=None)
    raw.columns = [str(c).strip() for c in raw.iloc[2]]
    data = raw.iloc[3:].reset_index(drop=True)

    # Índices fixos (mais robustos que busca por nome)
    I_PAT=0; I_FDUR=6; I_FILE=7; I_FONS=9; I_FOFF=10; I_NSEIZ=11; I_SON=12; I_SDUR=13

    out = {}
    cur_pat, cur_file, cur_fons = None, None, None

    for _, row in data.iterrows():
        # Paciente
        if pd.notna(row.iloc[I_PAT]):
            try: cur_pat = f"p{int(float(row.iloc[I_PAT]))}"
            except: pass

        # Arquivo
        if pd.notna(row.iloc[I_FILE]):
            cur_file = str(row.iloc[I_FILE]).strip()
            if not cur_file.lower().endswith('.edf'):
                cur_file += '.edf'
            cur_fons = _time_to_sec(row.iloc[I_FONS])

        if cur_pat is None or cur_file is None:
            continue
        if patients is not None and cur_pat not in patients:
            continue

        # Onset da crise
        son = _time_to_sec(row.iloc[I_SON])
        if son is None:
            continue

        # Duração como texto 'X min, Y sec' ou 'X min Y sec' ou 'Y sec'
        dsec = 0.0
        dtxt = str(row.iloc[I_SDUR])
        mm = re.search(r'(\d+)\s*min', dtxt)
        ss = re.search(r'(\d+)\s*sec', dtxt)
        if mm: dsec += int(mm.group(1)) * 60
        if ss: dsec += int(ss.group(1))
        if dsec <= 0:
            continue

        # Onset relativo ao início do arquivo (trata cruzamento de meia-noite)
        onset_rel = son - (cur_fons or 0.0)
        if onset_rel < 0:
            onset_rel += 86400
        onset_rel = max(0.0, onset_rel)

        out.setdefault(cur_pat, {}).setdefault(cur_file, []).append(
            (onset_rel, onset_rel + dsec)
        )
    return out

_mend_xlsx = os.path.join(MENDELEY_DIR, 'Documentation', 'Seizures_Information.xlsx')
mendeley_ann_all = {}
if os.path.exists(_mend_xlsx):
    parsed = parse_mendeley_xlsx(_mend_xlsx, PATIENTS['Mendeley'])
    for p in PATIENTS['Mendeley']:
        mendeley_ann_all[p] = parsed.get(p, {})
        print(f"✅ Mendeley {p} — {len(mendeley_ann_all[p])} arquivos com crise")
        for fn, seiz in mendeley_ann_all[p].items():
            for s, e in seiz:
                print(f"   {fn}  →  {s:.0f}s – {e:.0f}s  ({e-s:.0f}s)")
else:
    print("⚠️  Seizures_Information.xlsx não encontrado — rode o download antes.")

✅ Mendeley p10 — 2 arquivos com crise
   p10_Record1.edf  →  7199s – 7644s  (445s)
   p10_Record2.edf  →  7200s – 7505s  (305s)
✅ Mendeley p11 — 4 arquivos com crise
   p11_Record1.edf  →  4205s – 4269s  (64s)
   p11_Record2.edf  →  6372s – 6423s  (51s)
   p11_Record3.edf  →  1517s – 1608s  (91s)
   p11_Record3.edf  →  4397s – 4480s  (83s)
   p11_Record3.edf  →  7201s – 7277s  (76s)
   p11_Record3.edf  →  9243s – 9316s  (73s)
   p11_Record4.edf  →  7177s – 8535s  (1358s)
✅ Mendeley p12 — 3 arquivos com crise
   p12_Record1.edf  →  5756s – 5832s  (76s)
   p12_Record1.edf  →  7194s – 7298s  (104s)
   p12_Record2.edf  →  7202s – 7320s  (118s)
   p12_Record2.edf  →  10177s – 10259s  (82s)
   p12_Record3.edf  →  188s – 270s  (82s)
   p12_Record3.edf  →  2654s – 2818s  (164s)
   p12_Record3.edf  →  7226s – 7339s  (113s)
✅ Mendeley p13 — 4 arquivos com crise
   p13_Record1.edf  →  7196s – 7248s  (52s)
   p13_Record2.edf  →  3241s – 3266s  (25s)
   p13_Record2.edf  →  7196s – 7212s  (16s)
   p

## 4 — Carregamento de EDF + reamostragem para fs comum

Siena é 512 Hz e Mendeley 500 Hz; os demais 256 Hz. Reamostramos **tudo para 256 Hz** para que as features espectrais sejam comparáveis e para poder juntar datasets no treino.

In [13]:
_NON_EEG = {'ecg','emg','acc','gyr','mov','resp','ekg','eog','chest','abd',
            'eda','temp','bvp','hr','ppg'}

def load_edf(path, scale_uv=True):
    """Carrega EDF, mantém só canais EEG. Retorna (data, sfreq, ch_names, dur_s)."""
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    try:
        raw.pick('eeg')
    except Exception:
        keep = [c for c in raw.ch_names if not any(p in c.lower() for p in _NON_EEG)]
        if keep: raw.pick(keep)
    data  = raw.get_data() * (1e6 if scale_uv else 1.0)
    sfreq = float(raw.info['sfreq'])
    return data, sfreq, list(raw.ch_names), data.shape[1] / sfreq

def resample_to(data, sfreq, target=TARGET_FS):
    """Reamostra (n_ch, n_samp) para target Hz via polifase. Mantém se já igual."""
    if int(sfreq) == int(target):
        return data, float(target)
    from math import gcd
    up, down = target, int(round(sfreq))
    g = gcd(up, down); up //= g; down //= g
    out = resample_poly(data, up, down, axis=1)
    return out.astype(np.float32), float(target)

### 4.1 Inventário de canais e regiões

Confira aqui se o mapa eletrodo→região cobre os canais reais de cada dataset. Ajuste `ELECTRODE_REGION` na config se algum canal aparecer sem região.

In [15]:
def _first_valid_edf(dataset):
    """Retorna o primeiro EDF válido (legível pelo MNE) do dataset."""
    if dataset == 'CHBMIT':
        p = PATIENTS['CHBMIT'][0]; d = os.path.join(CHBMIT_DIR, p)
    elif dataset == 'Siena':
        p = PATIENTS['Siena'][0]; d = os.path.join(SIENA_DIR, p)
    elif dataset == 'Mendeley':
        d = os.path.join(MENDELEY_DIR, 'Raw_EDF_Files')
    elif dataset == 'SeizeIT2':
        p = PATIENTS['SeizeIT2'][0]; d = os.path.join(SEIZEIT_DIR, p, SEIZEIT_SESSION, 'eeg')
    else:
        return None
    if not os.path.isdir(d): return None
    # filtra .part (downloads incompletos) e arquivos muito pequenos (<1MB)
    edfs = sorted(f for f in os.listdir(d)
                  if f.lower().endswith('.edf') and not f.endswith('.part')
                  and os.path.getsize(os.path.join(d, f)) > 1_000_000)
    for fname in edfs:
        path = os.path.join(d, fname)
        try:
            mne.io.read_raw_edf(path, preload=False, verbose=False)
            return path   # primeiro que abriu sem erro
        except Exception:
            print(f"  ⚠️  {fname} inválido ou corrompido — pulando")
    return None

for ds in ['CHBMIT', 'Siena', 'Mendeley', 'SeizeIT2']:
    path = _first_valid_edf(ds)
    if not path:
        print(f"⚠️  {ds}: nenhum EDF válido encontrado ainda.")
        continue
    try:
        _, sf, ch, _ = load_edf(path)
        reg = regions_present(ch, ds)
        print(f"\n▸ {ds}  (fs={sf:.0f}Hz, {len(ch)} canais)  [{os.path.basename(path)}]")
        print(f"   canais: {ch[:8]}{' ...' if len(ch)>8 else ''}")
        print("   regiões:", {r: len(idx) for r, idx in reg.items() if idx})
    except Exception as e:
        print(f"⚠️  {ds}: erro ao carregar {os.path.basename(path)}: {e}")


▸ CHBMIT  (fs=256Hz, 23 canais)  [chb01_03.edf]
   canais: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1'] ...
   regiões: {'frontal': 9, 'temporal': 9, 'central': 3, 'parietal': 2}

▸ Siena  (fs=512Hz, 35 canais)  [PN00-1.edf]
   canais: ['EEG Fp1', 'EEG F3', 'EEG C3', 'EEG P3', 'EEG O1', 'EEG F7', 'EEG T3', 'EEG T5'] ...
   regiões: {'frontal': 7, 'temporal': 4, 'central': 7, 'parietal': 3, 'occipital': 2}

▸ Mendeley  (fs=500Hz, 19 canais)  [p10_Record1.edf]
   canais: ['EEG Fp1-Ref', 'EEG Fp2-Ref', 'EEG F3-Ref', 'EEG F4-Ref', 'EEG C3-Ref', 'EEG C4-Ref', 'EEG P3-Ref', 'EEG P4-Ref'] ...
   regiões: {'frontal': 7, 'temporal': 4, 'central': 2, 'parietal': 2, 'occipital': 2}

▸ SeizeIT2  (fs=256Hz, 2 canais)  [sub-001_ses-01_task-szMonitoring_run-03_eeg.edf]
   canais: ['BTEleft SD', 'CROSStop SD']
   regiões: {'temporal': 2}


## 5 — Pré-processamento + Rotulagem + Salvamento Nível 1 (L1)

Notch (60Hz CHB-MIT, 50Hz demais) → passa-alta 0.5Hz → passa-baixa 40Hz. Rotula cada amostra e salva o L1 por arquivo.

In [16]:
import os
import gc
import json
import time
import numpy as np
from math import gcd
from scipy.signal import iirnotch, filtfilt, butter, sosfiltfilt, resample_poly
from scipy.ndimage import uniform_filter1d

# --- 1. Funções de Filtragem e Rótulos ---
def notch_filter(data, sfreq, freq, Q=30):
    b, a = iirnotch(freq, Q, sfreq); return filtfilt(b, a, data, axis=-1)

def highpass_filter(data, sfreq, cutoff=F_HP, order=F_ORDER):
    sos = butter(order, cutoff, btype='high', fs=sfreq, output='sos'); return sosfiltfilt(sos, data, axis=-1)

def lowpass_filter(data, sfreq, cutoff=F_LP, order=F_ORDER):
    sos = butter(order, cutoff, btype='low', fs=sfreq, output='sos'); return sosfiltfilt(sos, data, axis=-1)

def preprocess(data, sfreq, notch_freq):
    out = notch_filter(data, sfreq, notch_freq)
    out = highpass_filter(out, sfreq)
    out = lowpass_filter(out, sfreq)
    return out

def build_label_array(n_samples, sfreq, seizure_intervals,
                      pre_sec=None, post_sec=POST_SEC, igap_sec=IGAP_SEC,
                      min_pre_sec=None):
    if pre_sec is None:      pre_sec = PRE_SEC
    if min_pre_sec is None:  min_pre_sec = globals().get('MIN_PRE_SEC', 0)

    labels   = np.full(n_samples, LBL['unknown'], dtype=np.int8)
    pre_n    = int(pre_sec * sfreq)
    post_n   = int(post_sec * sfreq)
    minpre_n = int(min_pre_sec * sfreq)

    intervals = sorted(seizure_intervals, key=lambda x: x[0])
    onsets  = [int(s * sfreq) for s, _ in intervals]
    offsets = [min(int(e * sfreq), n_samples) for _, e in intervals]

    for i_s, i_e in zip(onsets, offsets):
        labels[i_s:i_e] = LBL['ictal']

    for k, i_s in enumerate(onsets):
        win_start = max(0, i_s - pre_n)
        if k > 0:
            win_start = max(win_start, offsets[k-1])
        if (i_s - win_start) < minpre_n:
            continue
        seg = labels[win_start:i_s]
        seg[seg == LBL['unknown']] = LBL['preictal']

    for k, i_e in enumerate(offsets):
        win_end = min(n_samples, i_e + post_n)
        if k < len(onsets) - 1:
            win_end = min(win_end, onsets[k+1])
        seg = labels[i_e:win_end]
        seg[(seg == LBL['unknown']) | (seg == LBL['preictal'])] = LBL['postictal']

    ev  = (labels != LBL['unknown']).astype(np.float32)
    gap = int(igap_sec * sfreq)
    near = (uniform_filter1d(ev, size=gap*2+1, mode='constant') > 0) if (gap>0 and ev.any()) else ev.astype(bool)
    labels[(~near) & (labels == LBL['unknown'])] = LBL['interictal']
    return labels

def save_level1(data_filt, labels, sfreq, ch_names, dataset, patient, fname, out_dir=L1_DIR):
    key  = f'{dataset}__{patient}__{os.path.splitext(os.path.basename(fname))[0]}'
    npz  = os.path.join(out_dir, key + '_L1.npz')
    if not os.path.exists(npz):
        np.savez_compressed(npz, data=data_filt.astype(np.float32), labels=labels,
                            sfreq=np.float32(sfreq), ch_names=np.array(ch_names),
                            dataset=dataset)
        print(f"💾 L1: {os.path.basename(npz)}")
    else:
        print(f"⏭️  Já existe: {os.path.basename(npz)}")
    return npz

def edf_root(dataset, patient):
    if dataset=='CHBMIT':   return os.path.join(CHBMIT_DIR, patient)
    if dataset=='Siena':    return os.path.join(SIENA_DIR, patient)
    if dataset=='Mendeley': return os.path.join(MENDELEY_DIR, 'Raw_EDF_Files')
    if dataset=='SeizeIT2': return os.path.join(SEIZEIT_DIR, patient, SEIZEIT_SESSION, 'eeg')

# --- 2. EXTRAÇÃO BLINDADA CONTRA OOM ---
_NON_EEG = {'ecg','emg','acc','gyr','mov','resp','ekg','eog','chest','abd','eda','temp','bvp','hr','ppg'}

def _validate_edf(path):
    """Verifica se o EDF tem header válido sem carregar os dados."""
    import mne
    try:
        r = mne.io.read_raw_edf(path, preload=False, verbose=False)
        r.close()
        return True
    except Exception:
        return False

def process_edf_efficiently(path, target_fs, notch_freq, scale_uv=True):
    """
    Lê o EDF canal por canal para evitar OOM em EDFs grandes (1-2 GB).
    """
    import mne
    raw = mne.io.read_raw_edf(path, preload=False, verbose=False)
    try:
        raw.pick('eeg')
    except Exception:
        keep = [c for c in raw.ch_names if not any(p in c.lower() for p in _NON_EEG)]
        if keep: raw.pick(keep)

    orig_sfreq = float(raw.info['sfreq'])
    ch_names   = list(raw.ch_names)
    n_ch       = len(ch_names)
    n_times    = raw.n_times

    up, down = target_fs, int(round(orig_sfreq))
    g = gcd(up, down); up //= g; down //= g

    target_n_times = int(np.ceil(n_times * up / down))
    final_data     = np.empty((n_ch, target_n_times), dtype=np.float32)
    chunk_sz       = 500_000

    t0 = time.time()
    for i, ch_name in enumerate(ch_names):
        elapsed = time.time() - t0
        print(f'\r  canal {i+1}/{n_ch} ({ch_name})  {elapsed:.0f}s decorridos',
              end='', flush=True)

        ch_data_raw = np.empty(n_times, dtype=np.float32)
        for start in range(0, n_times, chunk_sz):
            stop = min(start + chunk_sz, n_times)
            chunk, _ = raw[i, start:stop]
            ch_data_raw[start:stop] = chunk[0] * (1e6 if scale_uv else 1.0)

        if orig_sfreq != target_fs:
            ch_data_proc = resample_poly(ch_data_raw, up, down).astype(np.float32)
        else:
            ch_data_proc = ch_data_raw
        del ch_data_raw

        ch_data_proc = preprocess(ch_data_proc, float(target_fs), notch_freq)

        min_len = min(target_n_times, len(ch_data_proc))
        final_data[i, :min_len] = ch_data_proc[:min_len]
        if min_len < target_n_times:
            final_data[i, min_len:] = 0
        del ch_data_proc

    print()
    raw.close()
    del raw
    gc.collect()

    return final_data, float(target_fs), ch_names


# --- 3. Execução Principal ---
l1_index = {}

def _resolve_edf_path(root_dir, fname):
    """
    Procura o EDF no diretório, tentando variações de nome:
      1. Nome exato
      2. Case-insensitive
      3. Substituição O/o ↔ 0 (typo Siena)
      4. Prefixo + sufixo numérico ('PN01.edf' → 'PN01-1.edf')
    Retorna o caminho real no disco, ou None se não achar.
    """
    if not os.path.isdir(root_dir):
        return None
    disk_files = [f for f in os.listdir(root_dir) if f.lower().endswith('.edf')]
    disk_lower = {f.lower(): f for f in disk_files}

    # 1. match exato
    p = os.path.join(root_dir, fname)
    if os.path.exists(p):
        return p

    # 2. case-insensitive
    if fname.lower() in disk_lower:
        return os.path.join(root_dir, disk_lower[fname.lower()])

    # 3. O/o ↔ 0 (typo Siena)
    fname_norm = re.sub(r'(PN)[Oo](\d)', lambda m: m.group(1)+'0'+m.group(2), fname)
    if fname_norm.lower() in disk_lower:
        return os.path.join(root_dir, disk_lower[fname_norm.lower()])

    # 4. prefixo numerado ('PN01.edf' → 'PN01-1.edf')
    base = os.path.splitext(fname_norm)[0]
    cands = sorted(f for f in disk_files if f.lower().startswith(base.lower() + '-'))
    if cands:
        return os.path.join(root_dir, cands[0])

    return None


def run_l1(dataset, patient, ann_dict):
    for fname, seiz in ann_dict.items():
        # Resolve o caminho real do EDF no disco
        root_dir = edf_root(dataset, patient)
        path = _resolve_edf_path(root_dir, fname)

        if path is None:
            print(f"  ⚠️  {fname} não encontrado em {root_dir}")
            continue

        # Usa o nome REAL do disco como chave (não o nome do txt)
        real_fname = os.path.basename(path)
        key = f'{dataset}__{patient}__{os.path.splitext(real_fname)[0]}'
        npz = os.path.join(L1_DIR, key + '_L1.npz')

        if os.path.exists(npz):
            print(f"⏭️  Já existe: {os.path.basename(npz)}")
            l1_index[key] = npz
            continue

        # Valida o header antes de processar
        if not _validate_edf(path):
            size_mb = os.path.getsize(path) / 1024 / 1024
            print(f"  ❌ {real_fname} CORROMPIDO ({size_mb:.1f} MB) — apagando")
            print(f"     → Rode a célula de download deste dataset para baixar novamente.")
            try: os.remove(path)
            except: pass
            continue

        print(f"\n⚙️  {dataset}/{patient}/{real_fname}")
        t0 = time.time()
        try:
            data_f, sfreq_out, ch = process_edf_efficiently(path, TARGET_FS, NOTCH[dataset])
        except Exception as e:
            print(f"  ❌ Erro ao processar {real_fname}: {e}")
            continue
        print(f"   processado em {time.time()-t0:.1f}s — shape {data_f.shape}")

        labels    = build_label_array(data_f.shape[1], sfreq_out, seiz)
        npz_salvo = save_level1(data_f, labels, sfreq_out, ch, dataset, patient, real_fname)
        l1_index[key] = npz_salvo

        del data_f, labels
        gc.collect()

ANN = {'CHBMIT': chbmit_ann_all, 'Siena': siena_ann_all,
       'Mendeley': mendeley_ann_all, 'SeizeIT2': seizeit_ann_all}

for ds, ann_all in ANN.items():
    for pat, ann in ann_all.items():
        if ann: run_l1(ds, pat, ann)

print(f"\n📦 Total L1: {len(l1_index)}")
json.dump({k: os.path.basename(v) for k,v in l1_index.items()},
          open(os.path.join(L1_DIR, '_index.json'),'w'), indent=2)


⏭️  Já existe: CHBMIT__chb01__chb01_03_L1.npz
⏭️  Já existe: CHBMIT__chb01__chb01_04_L1.npz
⏭️  Já existe: CHBMIT__chb01__chb01_15_L1.npz
⏭️  Já existe: CHBMIT__chb01__chb01_16_L1.npz
⏭️  Já existe: CHBMIT__chb01__chb01_18_L1.npz
⏭️  Já existe: CHBMIT__chb01__chb01_21_L1.npz
⏭️  Já existe: CHBMIT__chb01__chb01_26_L1.npz
⏭️  Já existe: CHBMIT__chb03__chb03_01_L1.npz
⏭️  Já existe: CHBMIT__chb03__chb03_02_L1.npz
⏭️  Já existe: CHBMIT__chb03__chb03_03_L1.npz
⏭️  Já existe: CHBMIT__chb03__chb03_04_L1.npz
⏭️  Já existe: CHBMIT__chb03__chb03_34_L1.npz
⏭️  Já existe: CHBMIT__chb03__chb03_35_L1.npz
⏭️  Já existe: CHBMIT__chb03__chb03_36_L1.npz
⏭️  Já existe: CHBMIT__chb04__chb04_05_L1.npz
⏭️  Já existe: CHBMIT__chb04__chb04_08_L1.npz
⏭️  Já existe: CHBMIT__chb04__chb04_28_L1.npz
⏭️  Já existe: CHBMIT__chb05__chb05_06_L1.npz
⏭️  Já existe: CHBMIT__chb05__chb05_13_L1.npz
⏭️  Já existe: CHBMIT__chb05__chb05_16_L1.npz
⏭️  Já existe: CHBMIT__chb05__chb05_17_L1.npz
⏭️  Já existe: CHBMIT__chb05__chb0

---
✅ **Fim do Notebook 1.** O L1 está em `data/level1_signals/`. Prossiga para o **Notebook 2 — Janelas, Níveis de Canal e Features**.